In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torchvision
from tqdm import tqdm

import pickle
from matplotlib import pyplot as plt

## NES

参考原文，实现NES黑盒对抗样本攻击

In [ ]:
from Week567_General_Code_Question import LeNet5, nes, evaluate
from Week567_General_Code_Question import visualize_adv

### 目标模型加载

注意，请将week5保存的lenet5.pt迁移到/model路径下

In [ ]:
model = LeNet5()
### 请将Week5保存的lenet5.pt迁移到/model路径下
model.load_state_dict(torch.load('model/lenet5.pt'))
model.eval()

criterion = nn.CrossEntropyLoss()

### 读入待攻击的样本

注意，需导入week6的新版本Week567_img_label.pkl

In [ ]:
with open('data/Week567_img_label.pkl', 'rb') as f:
    data = pickle.load(f)
    imgs, labels = data['img'], data['label']

### 实现基于NES梯度估计的攻击

- 请在Week567_General_Code_Question.py的`nes(imgs, epsilon, model, labels, sigma, n)`函数中实现NES黑盒梯度估计
  * 核心思想：根据导数定义，对于$u \to 0$，有 $\nabla_x f(x) = \frac{f(x+u) - f(x-u)}{2u}$；
  * 给定黑盒函数 $f$，单个样本 $x \in \mathbb{R}^N$，采样次数 $n$，搜索标准差 $\sigma$，梯度 $\nabla_x f(x)$ 可以通过如下方法估计：
      1. 采样噪声 $u_i \leftarrow \mathcal{N}(0_N, I_{N \cdot N})$
      2. 估计梯度 $g_i \leftarrow \frac{f(x+\sigma \cdot u_i) - f(x-\sigma \cdot u_i)}{2\sigma}$
      3. 对于 $u_i$ 做 $n$ 次采样，求均值，$\nabla_x f(x) \approx \frac{1}{n} \sum_{i=1}^n g_i$
  * 估计到的梯度，可用于 FGSM / PGD 算法以构造对抗样本

- 建议使用的API：
  - torch.randn [Link](https://pytorch.org/docs/stable/generated/torch.randn.html)
    - 采样随机高斯噪声

In [ ]:
epsilon = 0.2
sigma = 0.001
n = 50  # NOTE：该参数可自行调整
adv_xs = nes(imgs, epsilon, model, labels, sigma, n)

### 评测攻击效果

- 评估模型对样本adv_xs的预测结果与真实标签labels的匹配率。该匹配率越低，则攻击效果越好。

In [ ]:
pred_label = evaluate(adv_xs, labels, model)

### 对抗样本可视化

In [ ]:
adv_imgs = adv_xs.reshape_as(imgs)
visualize_adv(adv_imgs, labels, pred_label)